## Running Spark Locally vs. in Production

Spark runs locally in this workshop, but in production you would typically use a managed platform such as Databricks. It is important to understand what happens under the hood when you use Spark on Databricks. Databricks provides a cluster that is already running, with cores allocated and driver memory configured. It also offers serverless compute, which scales automatically as your jobs run.

## Creating the SparkSession

Below, we create a `SparkSession` that uses all the cores available on your local machine, with a driver memory of `6g`, which is enough for this workshop. If your machine has 4 cores, Spark has 4 threads to split tasks across and run in parallel.

Driver memory is the amount of memory allocated to the driver process of this Spark application. In local mode, the driver also runs the tasks, so this setting effectively caps how much data Spark can hold in memory. Make sure you do not allocate too little memory when processing large volumes of data.

## Configuring Shuffle Partitions

The `spark.sql.shuffle.partitions` setting controls how many partitions Spark creates after a shuffle, such as in a join, `groupBy`, or other wide transformation. The default is `200`, which is excessive for a single machine.

- **Too low:** each partition holds more data, which increases memory pressure and may leave some cores idle (if there are fewer partitions than cores).
- **Too high:** Spark spends more time scheduling many tiny tasks than doing useful work.


In [1]:
from pyspark.sql import SparkSession 
import os 

N_CORES = os.cpu_count()

spark = (SparkSession.builder.appName("hands_on")
         .master(f"local[{N_CORES}]")
         .config("spark.driver.memory", "6g")
         .config("spark.sql.shuffle.partitions",N_CORES * 2)  
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print(f"Cores available: {N_CORES}")
print(f"Shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/27 03:55:12 WARN Utils: Your hostname, Tusche, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/27 03:55:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/27 03:55:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Cores available: 4
Shuffle partitions: 8


## About the Data

This workshop uses dimension and fact tables to mirror the real-world scenarios you will encounter in production. We create four tables:

- **`dim_users` (dimension table):** a small table, small enough to broadcast.
- **`fact_transactions` (fact table A):** a large table that joins to `dim_users`, with `user_id` evenly distributed. It has 5M rows, which is safe for a 4-core laptop with modest memory, yet large enough to make a shuffle join noticeably slower than a broadcast join.
- **`fact_events` (fact table B):** a large table that joins to `fact_transactions` on `txn_id`. Because both sides are large, this forces a true shuffle (sort-merge) join.
- **`fact_transactions_skewed` (skewed variant):** a copy of `fact_transactions` in which one `user_id` holds about 30% of the rows. It demonstrates how data skew affects a sort-merge join. 


In [2]:
from pyspark.sql import functions as F

dim_users = (
    spark.range(2_000)
    .withColumnRenamed("id", "user_id")
    .withColumn("user_name", F.concat(F.lit("user_"), F.col("user_id")))
    .withColumn("signup_region", F.element_at(
        F.array(*[F.lit(r) for r in ["NG", "US", "UK", "DE", "IN"]]),
        (F.col("user_id") % 5 + 1).cast("int"),
    ))
)


fact_transactions = (
    spark.range(5_000_000)
    .withColumnRenamed("id", "txn_id")
    .withColumn("user_id", (F.col("txn_id") % 2_000).cast("long"))
    .withColumn("amount", F.round(F.rand(seed=1) * 500, 2))
)

fact_txn_events = (
    spark.range(5_000_000)
    .withColumnRenamed("id", "txn_id")
    .withColumn("device_type", F.element_at(
        F.array(*[F.lit(d) for d in ["mobile", "desktop", "tablet"]]),
        (F.col("txn_id") % 3 + 1).cast("int"),
    ))
    .withColumn("session_seconds", (F.rand(seed=2) * 600).cast("int"))
)


fact_transactions_skewed = (
    spark.range(5_000_000)
    .withColumnRenamed("id", "txn_id")
    .withColumn(
        "user_id",
        F.when(F.rand(seed=3) < 0.30, F.lit(9999))          # ~30% of rows pinned to one key
         .otherwise((F.col("txn_id") % 2_000).cast("long")),
    )
    .withColumn("amount", F.round(F.rand(seed=4) * 500, 2))
)

for name, df in [
    ("dim_users", dim_users),
    ("fact_transactions", fact_transactions),
    ("fact_txn_events", fact_txn_events),
    ("fact_transactions_skewed", fact_transactions_skewed),
]:
    print(f"{name}: {df.count():,} rows")

dim_users: 2,000 rows
fact_transactions: 5,000,000 rows
fact_txn_events: 5,000,000 rows
fact_transactions_skewed: 5,000,000 rows


## Broadcast Join

A broadcast join sends a full copy of the small side of the join (the dimension table) to every core, while the large table is split into partitions across those cores. In this workshop, with four cores, each core holds the entire dimension table but only one partition of the large table. This makes the join fast because Spark avoids shuffling the large table. However, it has limitations: the broadcast side must be small enough to fit in memory on the driver and on every executor.

`spark.sql.autoBroadcastJoinThreshold` controls Spark's automatic choice. If Spark's size estimate for one side of the join is below this threshold (10 MB by default), Spark broadcasts that side automatically, with no hint needed. This estimate can be wrong, for example when table statistics are stale or when a filtered DataFrame cannot be sized accurately. For this reason, production jobs do not always rely on it and often use an explicit `broadcast()` hint instead.

Note `spark.sql.autoBroadcastJoinThreshold` has a defualt value of 10 MB. This means if any of the file is lower or equals to 10 MB it automaticaly broadcast but you can change this setting if you wish to.

In [3]:
# here means automatically broadcast with 10mb threshold 
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

# here means automatically broadcast with threshold changed to 50mb (this code is not ran for the broadcast 
# section purpose)
# spark.conf.get("spark.sql.autoBroadcastJoinThreshold", 50 * 1024 * 1024)


# here means do not automatically broadcast unless I ask you to, use shuffle instead (this code is not ran for the broadcast 
# section purpose)
# spark.conf.get("spark.sql.autoBroadcastJoinThreshold", -1)


'10485760b'

## Broadcast Join: What It Can and Cannot Do

Here we broadcast the dimension table and partition the fact table, so the query is a left join from the fact table to the dimension table, with the dimension table broadcast. This works because we only need the dimension rows that match the fact rows. Every fact row is preserved, and each one is looked up against the dimension table.

Now suppose some dimension rows have no match in the fact table, and we want them to appear in the result. A broadcast join cannot do this. To see why, remember how the work is distributed: each core holds a full copy of the dimension table, but no core ever sees the complete fact table, only its own partition.

Each core performs its broadcast join as a completely local, independent operation. It takes its partition of the fact table, probes each row against its full copy of the dimension table, and emits the matches. For the preserved fact side, it also emits any unmatched rows from its own partition. Cores never communicate during this step, which is the point of broadcasting: it avoids the network shuffle.

That independence is exactly why unmatched dimension rows are a problem. To know that a dimension row has no match, you would need to ask whether any fact row, in any partition, matches its key. No single core can answer that alone. One core may see no match for `user_id = 42` in its partition, while another core's partition contains the one fact row that matches it. Confirming that a dimension row is globally unmatched requires combining results across all partitions. That coordination is itself a shuffle, which is the cost a broadcast join exists to avoid.

In practice, Spark cannot use a broadcast hash join when the broadcast side is the preserved side of an outer join (for example, a right outer join where the dimension table is on the right and is the side being preserved, or a full outer join). It falls back to another strategy, typically a sort-merge join.

In [4]:
joined = fact_transactions.join(F.broadcast(dim_users), on = "user_id", how = "inner")
joined.show(20)
joined.explain()

+-------+------+------+---------+-------------+
|user_id|txn_id|amount|user_name|signup_region|
+-------+------+------+---------+-------------+
|      0|     0|318.19|   user_0|           NG|
|      1|     1|299.69|   user_1|           US|
|      2|     2| 67.42|   user_2|           UK|
|      3|     3| 38.42|   user_3|           DE|
|      4|     4|426.96|   user_4|           IN|
|      5|     5|358.39|   user_5|           NG|
|      6|     6| 123.7|   user_6|           US|
|      7|     7| 68.37|   user_7|           UK|
|      8|     8|193.48|   user_8|           DE|
|      9|     9|302.58|   user_9|           IN|
|     10|    10|275.65|  user_10|           NG|
|     11|    11| 72.62|  user_11|           US|
|     12|    12|130.82|  user_12|           UK|
|     13|    13|426.51|  user_13|           DE|
|     14|    14|404.51|  user_14|           IN|
|     15|    15|482.16|  user_15|           NG|
|     16|    16|365.61|  user_16|           US|
|     17|    17|475.35|  user_17|       

## Forcing a Broadcast on a Full Outer Join

Here we force a broadcast hint on a full outer join, which keeps the unmatched rows from both sides. The query runs successfully, but Spark does not follow the hint. Instead, it falls back to a sort-merge join, because a broadcast hash join cannot produce a correct result for a full outer join.

A broadcast hint is a suggestion to the optimizer, not a command. When the requested strategy cannot guarantee correct results, Spark ignores the hint without raising an error. You can confirm this by calling `.explain()` on the result and checking the physical plan: it shows `SortMergeJoin` instead of `BroadcastHashJoin`.

> **Note: this fallback applies to equi-joins.** Every join in this workshop uses an equality condition (such as `user_id` or `txn_id`), so Spark can fall back to a sort-merge join. A non-equi join, which uses a condition like `<`, `>=`, or `BETWEEN`, has no key to hash or sort on. For a non-equi full outer join, Spark cannot use a sort-merge join. It typically chooses a `BroadcastNestedLoopJoin` instead, which compares every row on one side against every row on the other. This works for small broadcast sides, but it becomes very slow or runs out of memory when both sides are large.

In [5]:
joined = fact_transactions.join(F.broadcast(dim_users), on = "user_id", how = "full_outer")
joined.show(20)
joined.explain()

26/10/27 03:55:27 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build right for full outer join.
26/10/27 03:55:31 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build right for full outer join.
26/10/27 03:55:31 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build right for full outer join.


+-------+------+------+---------+-------------+
|user_id|txn_id|amount|user_name|signup_region|
+-------+------+------+---------+-------------+
|      2|     2| 67.42|   user_2|           UK|
|      2|  2002|206.71|   user_2|           UK|
|      2|  4002| 136.4|   user_2|           UK|
|      2|  6002|430.88|   user_2|           UK|
|      2|  8002|212.97|   user_2|           UK|
|      2| 10002|322.65|   user_2|           UK|
|      2| 12002| 136.5|   user_2|           UK|
|      2| 14002|446.88|   user_2|           UK|
|      2| 16002|479.04|   user_2|           UK|
|      2| 18002|498.45|   user_2|           UK|
|      2| 20002|156.67|   user_2|           UK|
|      2| 22002| 227.5|   user_2|           UK|
|      2| 24002|287.01|   user_2|           UK|
|      2| 26002| 30.81|   user_2|           UK|
|      2| 28002| 23.01|   user_2|           UK|
|      2| 30002|380.96|   user_2|           UK|
|      2| 32002|304.32|   user_2|           UK|
|      2| 34002|233.93|   user_2|       

26/10/27 03:55:32 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build right for full outer join.


## Shuffle Join on Two Large Fact Tables

Here we join two large fact tables with a full outer join. We disable automatic broadcasting by setting `spark.sql.autoBroadcastJoinThreshold` to `-1`. The query runs correctly, but Spark uses a sort-merge join, which shuffles both tables across the network. Shuffle joins are expensive and slow, because every row must be written to disk, moved between executors, and sorted before the join can happen.

Sometimes you need to take extra steps to make a join faster or cheaper. One approach is to break the join down yourself so that part of it can use a broadcast join. For example, a full outer join can be rewritten as a left join plus a left anti join (the rows from the other side that have no match), combined with a union. Each piece has a preserved side that is large and a lookup side that may be small enough to broadcast, such as the distinct join keys of one table.

In [6]:
import time 
def timed(df, label:"join"):
    
    now = time.time()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label}: {time.time() - now:.1f}s")


now = time.time()
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

joined = fact_transactions.join(fact_txn_events, on = "txn_id", how = "full_outer")


timed(joined, "broadcast rewrite")
joined.explain()

broadcast rewrite: 13.5s
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [coalesce(txn_id#5L, txn_id#9L) AS txn_id#77L, user_id#6L, amount#7, device_type#10, session_seconds#11]
   +- SortMergeJoin [txn_id#5L], [txn_id#9L], FullOuter
      :- Sort [txn_id#5L ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(txn_id#5L, 8), ENSURE_REQUIREMENTS, [plan_id=514]
      :     +- Project [id#4L AS txn_id#5L, (id#4L % 2000) AS user_id#6L, round((rand(1) * 500.0), 2) AS amount#7]
      :        +- Range (0, 5000000, step=1, splits=4)
      +- Sort [txn_id#9L ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(txn_id#9L, 8), ENSURE_REQUIREMENTS, [plan_id=515]
            +- Project [id#8L AS txn_id#9L, element_at([mobile,desktop,tablet], cast(((id#8L % 3) + 1) as int), None, true) AS device_type#10, cast((rand(2) * 600.0) as int) AS session_seconds#11]
               +- Range (0, 5000000, step=1, splits=4)




## Optimizing the Full Outer Join with Broadcast Joins

A full outer join cannot use a broadcast hash join directly, so Spark falls back to a sort-merge join that shuffles both tables. If one of the tables is small enough to broadcast, you can break the join down yourself and keep every step as a broadcast join.

A full outer join is equivalent to two parts combined with a union:

1. **A left join** from `fact_transactions` to `fact_txn_events`. This returns every transaction, with its matching events when they exist.
2. **A left anti join** from `fact_txn_events` to `fact_transactions`. This returns the events that have no matching transaction. Only the `txn_id` column of `fact_transactions` is needed here, so we broadcast just that column.

We then combine both parts with `unionByName`, using `allowMissingColumns=True`. The transaction columns are null for the event-only rows.

In the physical plan, you should see two `BroadcastHashJoin` nodes (`LeftOuter` and `LeftAnti`) under a `Union`, and no `SortMergeJoin`.

### Verify the Result

Confirm that the rewrite returns the same rows as the original full outer join. The row counts should match, and `exceptAll` in both directions should return zero rows.

### Limitations

- **The broadcast side must fit in memory.** Part 1 broadcasts the full `fact_txn_events` table, which is collected on the driver and copied to every executor. If it is too large, the job fails with an out-of-memory error where the shuffle version would succeed.
- **Each table is read twice.** This is usually cheaper than a shuffle, but it is still a real cost.
- **Duplicate column names cause problems.** If both tables share a non-key column name, such as `amount`, rename one before joining so that `unionByName` is not ambiguous.
- **Row order is not guaranteed.** The union does not preserve ordering, so `show(20)` may return different rows than the original join.
- **If both tables are truly large, the shuffle is hard to avoid.** This technique only helps when one side is small enough to broadcast.

In [7]:
from pyspark.sql import functions as F

def timed(df, label:"join"):
    
    now = time.time()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label}: {time.time() - now:.1f}s")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)


left_part = fact_transactions.join(
    F.broadcast(fact_txn_events), on="txn_id", how="left"
)


txn_keys = fact_transactions.select("txn_id")
right_only = fact_txn_events.join(
    F.broadcast(txn_keys), on="txn_id", how="left_anti"
)

joined_opt = left_part.unionByName(right_only, allowMissingColumns=True)


timed(joined_opt, "broadcast rewrite")
joined_opt.explain()


broadcast rewrite: 11.8s
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Union
   :- Project [txn_id#5L, user_id#6L, amount#7, device_type#10, session_seconds#11]
   :  +- BroadcastHashJoin [txn_id#5L], [txn_id#9L], LeftOuter, BuildRight, false, false
   :     :- Project [id#4L AS txn_id#5L, (id#4L % 2000) AS user_id#6L, round((rand(1) * 500.0), 2) AS amount#7]
   :     :  +- Range (0, 5000000, step=1, splits=4)
   :     +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=729]
   :        +- Project [id#8L AS txn_id#9L, element_at([mobile,desktop,tablet], cast(((id#8L % 3) + 1) as int), None, true) AS device_type#10, cast((rand(2) * 600.0) as int) AS session_seconds#11]
   :           +- Range (0, 5000000, step=1, splits=4)
   +- Project [txn_id#79L, null AS user_id#86L, null AS amount#87, device_type#80, session_seconds#81]
      +- BroadcastHashJoin [txn_id#79L], [txn_id#83L], LeftAnti, BuildRight, false, false
         :- Projec

### Comparing the Results

In our run, the optimized version finished faster than the original shuffle join. On very large datasets, a gap like this can translate into significant time and cost savings, as long as the broadcast side still fits in memory.

The two versions also behaved differently when we added `joined.show()`. On a 4-core laptop, the unoptimized version failed, most likely because the sort-merge join shuffles and sorts both tables, and `show()` triggers the job a second time. This puts heavy pressure on CPU and memory. The optimized version avoids the shuffle, so it completed without errors.

### Inspecting the Skew

The dataset below is heavily skewed. A quick look at the key counts shows that `user_id = 9999` appears 1,498,063 times, far more than any other key. Because a shuffle sends all rows with the same key to the same partition, a single task has to process all of those rows, which can slow down or stall the whole job.

In [8]:
counts = fact_transactions_skewed.groupby(F.col("user_id")).agg(F.count("*").alias("counts"))
counts.orderBy(F.col("counts").desc()).show(10)

+-------+-------+
|user_id| counts|
+-------+-------+
|   9999|1498063|
|   1996|   1828|
|   1743|   1826|
|    896|   1820|
|   1091|   1816|
|    487|   1811|
|   1987|   1811|
|   1670|   1810|
|   1521|   1808|
|    768|   1807|
+-------+-------+
only showing top 10 rows


## Handling Data Skew

Data skew occurs when one key appears far more often than the others. In our skewed dataset, a single `user_id` accounts for almost 60% of the rows. Because a shuffle sends all rows with the same key to the same partition, one task receives a disproportionate share of the data. The other tasks finish quickly and sit idle, waiting for that one slow task. The job is only as fast as its slowest task.

To handle skew, we will explore two approaches: Adaptive Query Execution (AQE) and salting. First, we need a baseline. We run a join against the skewed dataset with AQE turned off and record how long it takes.

In [ ]:
def timed(df, label):
    now = time.time()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label}: {time.time() - now:.1f}s")

spark.conf.set("spark.sql.adaptive.enabled", "false")

baseline = fact_transactions_skewed.join(dim_users, on="user_id", how="inner")
baseline.explain()
timed(baseline, "baseline (AQE off)")

== Physical Plan ==
*(5) Project [user_id#14L, txn_id#13L, amount#15, user_name#2, signup_region#3]
+- *(5) SortMergeJoin [user_id#14L], [user_id#1L], Inner
   :- *(2) Sort [user_id#14L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(user_id#14L, 8), ENSURE_REQUIREMENTS, [plan_id=822]
   :     +- *(1) Filter isnotnull(user_id#14L)
   :        +- *(1) Project [txn_id#13L, user_id#14L, round((rand(4) * 500.0), 2) AS amount#15]
   :           +- *(1) Project [id#12L AS txn_id#13L, CASE WHEN (rand(3) < 0.3) THEN 9999 ELSE (id#12L % 2000) END AS user_id#14L]
   :              +- *(1) Range (0, 5000000, step=1, splits=4)
   +- *(4) Sort [user_id#1L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(user_id#1L, 8), ENSURE_REQUIREMENTS, [plan_id=828]
         +- *(3) Project [id#0L AS user_id#1L, concat(user_, cast(id#0L as string)) AS user_name#2, element_at([NG,US,UK,DE,IN], cast(((id#0L % 5) + 1) as int), None, true) AS signup_region#3]
            +- *(3) Range (0,

baseline (AQE off): 19.6s


### Enabling AQE

With AQE turned on, the same join runs noticeably faster. AQE detects the skewed partition at runtime and splits it into smaller pieces, so no single task is given a disproportionate amount of work.

We also need to lower a few default settings. Spark's defaults for skew detection and partition sizing are tuned for production clusters, where partitions are hundreds of megabytes. On a laptop with a much smaller dataset, the skewed partition never crosses those thresholds, so AQE

In [10]:
def timed(df, label):
    now = time.time()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label}: {time.time() - now:.1f}s")

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "16MB")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "16MB")

joined = fact_transactions_skewed.join(dim_users, on="user_id", how="inner")
baseline.explain()
timed(joined, "baseline (AQE off)")

    

== Physical Plan ==
*(5) Project [user_id#14L, txn_id#13L, amount#15, user_name#2, signup_region#3]
+- *(5) SortMergeJoin [user_id#14L], [user_id#1L], Inner
   :- *(2) Sort [user_id#14L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(user_id#14L, 8), ENSURE_REQUIREMENTS, [plan_id=822]
   :     +- *(1) Filter isnotnull(user_id#14L)
   :        +- *(1) Project [txn_id#13L, user_id#14L, round((rand(4) * 500.0), 2) AS amount#15]
   :           +- *(1) Project [id#12L AS txn_id#13L, CASE WHEN (rand(3) < 0.3) THEN 9999 ELSE (id#12L % 2000) END AS user_id#14L]
   :              +- *(1) Range (0, 5000000, step=1, splits=4)
   +- *(4) Sort [user_id#1L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(user_id#1L, 8), ENSURE_REQUIREMENTS, [plan_id=828]
         +- *(3) Project [id#0L AS user_id#1L, concat(user_, cast(id#0L as string)) AS user_name#2, element_at([NG,US,UK,DE,IN], cast(((id#0L % 5) + 1) as int), None, true) AS signup_region#3]
            +- *(3) Range (0,

baseline (AQE off): 7.8s


In [ ]:
def timed(df, label):
    now = time.time()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label}: {time.time() - now:.1f}s")

spark.conf.set("spark.sql.adaptive.enabled", "false")

SALT_BUCKETS = 10

salted_fact = fact_transactions_skewed.withColumn(
    "salt", (F.rand(seed=42) * SALT_BUCKETS).cast("int")
)

# Other side: replicate each row once per salt value
salts = spark.range(SALT_BUCKETS).select(F.col("id").cast("int").alias("salt"))
salted_dim = dim_users.crossJoin(salts)

salted = (
    salted_fact.join(salted_dim, on=["user_id", "salt"], how="inner")
    .drop("salt")
)

salted.explain()
timed(salted, "salted")